# Phase 4 — Strategic Recommendations
**ITS 2122 — Python for Data Science & AI | Sem 3 2026**  
**Project:** Credit Risk & Customer Financial Health Analytics

---

## Objectives

A risk score is only useful once it is translated into action. This phase addresses three questions:

1. **Credit Utilisation & Exposure Hypothesis** — do customers using a large share of their credit limit, combined with weak repayment behaviour, contribute disproportionately to default risk?
2. **Financial Significance** — how much money (NTD) does each risk segment actually represent, and where is expected loss concentrated?
3. **Strategic Action Mapping** — what specific, fair, and transparent action should the bank take for each segment?

This notebook reproduces the Phase 3 scoring pipeline independently so it can run standalone. In the final merged submission notebook, this recomputation is unnecessary since `risk_segment` will already exist from Phase 3.

## Imports and Setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── Display preferences ───────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
os.makedirs('figures', exist_ok=True)
print("Libraries loaded ✓")


## Constants

In [ ]:
# ── File path ────────────────────────────────────────────────────
CLEANED_PATH = "cleaned_credit_data.csv"

# ── Column groups ────────────────────────────────────────────────
PAY_STATUS_COLS = [f'pay_status_{i}' for i in range(1, 7)]
BILL_AMOUNT_COLS = [f'bill_amount_{i}' for i in range(1, 7)]
PAY_AMOUNT_COLS = [f'pay_amount_{i}' for i in range(1, 7)]
TARGET = 'default_next_month'

# ── Ghost-account detection (from Phase 2/3) ──────────────────────
GHOST_RECENT_STATUS = 1
GHOST_INACTIVE_CODE = -2

# ── Phase 3 scoring constants (reproduced for standalone use) ─────
DELINQUENCY_BINS = [-1, 0, 1, 2, 4, 6]
DELINQUENCY_LABELS = [1, 2, 3, 4, 5]
REPAYMENT_LABELS = [5, 4, 3, 2, 1]
REPAYMENT_NAN_SCORE = 3
EXPOSURE_LABELS = [1, 2, 3, 4, 5]
TRAJECTORY_LABELS = [1, 2, 3, 4, 5]
DELINQUENCY_WEIGHT = 2
SECONDARY_WEIGHT = 1
SEGMENT_BINS = [4, 9, 14, 19, 25]
SEGMENT_LABELS = ['Healthy', 'Watchlist', 'At-Risk', 'Critical']
SEGMENT_ORDER = ['Healthy', 'Watchlist', 'At-Risk', 'Critical']
SEGMENT_COLOURS = {
    'Healthy':   '#2ca25f',
    'Watchlist': '#feb24c',
    'At-Risk':   '#f03b20',
    'Critical':  '#7a0177',
}

# ── Phase 4 hypothesis-test thresholds ─────────────────────────────
# 50% utilisation is a common industry rule of thumb: revolving
# balances above half the credit limit are considered elevated
# risk by most card issuers.
HIGH_UTIL_THRESHOLD = 0.50
# Paying back less than a fifth of the average bill signals
# materially weak repayment capacity.
LOW_RATIO_THRESHOLD = 0.20

print("Constants defined ✓")


---
## Load Data and Reproduce Phase 3 Segments

Full rationale for each scoring dimension is documented in the Phase 3 notebook; the pipeline is reproduced here compactly so this notebook can be tested independently.

In [ ]:
df = pd.read_csv(CLEANED_PATH)

assert df.shape == (30_000, 34), (
    f'Unexpected shape {df.shape}; expected (30000, 34).'
)

# ── Derived columns (Phase 2/3) ────────────────────────────────────
df['utilization'] = (
    df['avg_bill_amount'] / df['credit_limit']
).clip(lower=0)

# Outstanding balance exposure: negative bills (credit-balance
# customers) are clipped to 0 since they represent money owed TO
# the customer, not exposure the bank is carrying.
df['outstanding_exposure'] = df['avg_bill_amount'].clip(lower=0)

df['is_ghost_account'] = (
    (df['pay_status_1'] == GHOST_RECENT_STATUS)
    & df[BILL_AMOUNT_COLS].eq(0).all(axis=1)
    & df[PAY_AMOUNT_COLS].eq(0).all(axis=1)
    & df[
        ['pay_status_2', 'pay_status_3', 'pay_status_4',
         'pay_status_5', 'pay_status_6']
    ].eq(GHOST_INACTIVE_CODE).all(axis=1)
)

# ── Phase 3 dimension scores ────────────────────────────────────────
delinquency_score = pd.cut(
    df['num_delayed_months'], bins=DELINQUENCY_BINS,
    labels=DELINQUENCY_LABELS, right=True,
).astype(int)

repayment_score = pd.qcut(
    df['pay_to_bill_ratio'], q=5,
    labels=REPAYMENT_LABELS, duplicates='drop',
).astype(float).fillna(REPAYMENT_NAN_SCORE).astype(int)

exposure_score = pd.qcut(
    df['utilization'], q=5,
    labels=EXPOSURE_LABELS, duplicates='drop',
).astype(int)

trajectory_score = pd.qcut(
    df['balance_trend'], q=5,
    labels=TRAJECTORY_LABELS, duplicates='drop',
).astype(int)

df['total_risk_score'] = (
    DELINQUENCY_WEIGHT * delinquency_score
    + SECONDARY_WEIGHT * repayment_score
    + SECONDARY_WEIGHT * exposure_score
    + SECONDARY_WEIGHT * trajectory_score
)

df['risk_segment'] = pd.cut(
    df['total_risk_score'], bins=SEGMENT_BINS,
    labels=SEGMENT_LABELS, right=True, include_lowest=True,
).astype(str)

# Ghost-account override (see Phase 3 for full justification)
df.loc[df['is_ghost_account'], 'risk_segment'] = 'At-Risk'

OVERALL_DR = df[TARGET].mean() * 100

print(f"Segments reproduced ✓")
print(df['risk_segment'].value_counts().reindex(SEGMENT_ORDER))


---
## Section 4.1 — Credit Utilisation and Exposure Investigation

Before testing the interaction hypothesis, we first look at how utilisation, outstanding balance, and credit limit each vary independently across the four risk segments.

In [ ]:
# ── Fig 16: Utilisation, balance, and limit by segment ───────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

panels = [
    ('utilization', 'Utilisation Ratio', axes[0]),
    ('outstanding_exposure', 'Outstanding Balance (NTD)', axes[1]),
    ('credit_limit', 'Credit Limit (NTD)', axes[2]),
]

for col, label, ax in panels:
    sns.boxplot(
        data=df, x='risk_segment', y=col,
        hue='risk_segment', order=SEGMENT_ORDER,
        hue_order=SEGMENT_ORDER,
        palette=SEGMENT_COLOURS, legend=False,
        showfliers=False, linewidth=1.1, ax=ax,
    )
    ax.set_title(label, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('')
    ax.set_ylabel(label, fontsize=10)
    ax.tick_params(axis='x', labelsize=9, rotation=15)

fig.suptitle(
    'Fig 16 — Utilisation, Outstanding Balance, and Credit '
    'Limit by Risk Segment\n'
    '(box = IQR, whiskers = 1.5×IQR, outliers omitted)',
    fontsize=13, fontweight='bold', y=1.03,
)
plt.tight_layout()
plt.savefig(
    'figures/fig16_utilization_balance_limit.png',
    dpi=150, bbox_inches='tight',
)
plt.show()

print('Median values by segment:')
print(
    df.groupby('risk_segment', observed=True)[
        ['utilization', 'outstanding_exposure', 'credit_limit']
    ].median().reindex(SEGMENT_ORDER).round(2).to_string()
)


**Interpretation — Fig 16**  
Utilisation rises steadily from Healthy through Critical, confirming the scoring framework's exposure dimension is behaving as intended. Outstanding balance follows the same upward pattern. Credit limit, however, moves in the **opposite direction** — Critical customers hold noticeably smaller limits than Healthy customers. Together this means Critical customers are maxing out a *smaller* line of credit, while Healthy customers carry a *larger* limit they barely touch. Limit size alone is not the driver of risk — **how much of it is used, relative to what's granted, is what matters.**

---
## Section 4.2 — Testing the Utilisation × Repayment Hypothesis

The business mandate asks specifically whether customers combining **high utilisation** with **weak repayment behaviour** contribute disproportionately to default risk. Two reusable classifier functions split the portfolio into four quadrants to test this directly.

In [ ]:
def classify_utilization_band(utilization: pd.Series) -> pd.Series:
    """Classify customers into high/low utilisation bands.

    Uses a fixed 50% threshold rather than a quantile split so
    the bands carry a fixed, business-interpretable meaning
    (independent of how this particular portfolio happens to be
    distributed).

    Parameters
    ----------
    utilization : pd.Series
        avg_bill_amount / credit_limit, clipped at 0.

    Returns
    -------
    pd.Series of str
        'High Utilization (≥50%)' or 'Low Utilization (<50%)'.
    """
    return pd.Series(
        np.where(
            utilization >= HIGH_UTIL_THRESHOLD,
            'High Utilization (≥50%)',
            'Low Utilization (<50%)',
        ),
        index=utilization.index,
    )


def classify_repayment_band(
    pay_to_bill_ratio: pd.Series,
) -> pd.Series:
    """Classify customers into adequate/poor repayment bands.

    NaN ratios (credit-balance customers with zero or negative
    average bills) are explicitly preserved as NaN rather than
    silently defaulting to 'Adequate' — this dimension has no
    meaning for them and they must be excluded downstream.

    Parameters
    ----------
    pay_to_bill_ratio : pd.Series
        Ratio of avg payment to avg bill. May contain NaN.

    Returns
    -------
    pd.Series of str (with NaN preserved)
        'Poor Repayment (<20%)' or 'Adequate Repayment (≥20%)'.
    """
    band = pd.Series(
        np.where(
            pay_to_bill_ratio < LOW_RATIO_THRESHOLD,
            'Poor Repayment (<20%)',
            'Adequate Repayment (≥20%)',
        ),
        index=pay_to_bill_ratio.index,
    )
    return band.where(pay_to_bill_ratio.notna())


df['util_band'] = classify_utilization_band(df['utilization'])
df['ratio_band'] = classify_repayment_band(
    df['pay_to_bill_ratio']
)

# ── Exclude undefined-ratio customers from this analysis ────────
# (Same 1,071 credit-balance customers flagged in Phase 1's
# flag_credit_balance; pay-to-bill ratio has no meaning for them.)
excluded_n = df['ratio_band'].isna().sum()
quadrant_df = df.dropna(subset=['ratio_band']).copy()

print(f"Excluded (undefined repayment ratio): {excluded_n:,} rows")
print(f"Analysed: {len(quadrant_df):,} rows")


In [ ]:
# ── Build the 2x2 quadrant matrix ────────────────────────────────
UTIL_ORDER = ['Low Utilization (<50%)', 'High Utilization (≥50%)']
RATIO_ORDER = [
    'Adequate Repayment (≥20%)', 'Poor Repayment (<20%)'
]

pivot_default_rate = (
    quadrant_df.pivot_table(
        values=TARGET, index='util_band',
        columns='ratio_band', aggfunc='mean',
    )
    .reindex(index=UTIL_ORDER, columns=RATIO_ORDER)
    * 100
)
pivot_count = (
    quadrant_df.pivot_table(
        values='customer_id', index='util_band',
        columns='ratio_band', aggfunc='count',
    )
    .reindex(index=UTIL_ORDER, columns=RATIO_ORDER)
)

# Combined annotation: default rate % on top, n below
annot_labels = (
    pivot_default_rate.round(1).astype(str) + '%\n(n='
    + pivot_count.astype(str) + ')'
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot_default_rate,
    annot=annot_labels,
    fmt='',
    cmap='RdYlGn_r',
    vmin=10, vmax=32,
    linewidths=1.0,
    linecolor='white',
    annot_kws={'size': 12, 'weight': 'bold'},
    cbar_kws={'label': 'Default Rate (%)'},
    ax=ax,
)
ax.set_title(
    'Fig 17 — Default Rate by Utilisation × Repayment '
    'Quadrant',
    fontsize=13, fontweight='bold', pad=14,
)
ax.set_xlabel('Repayment Behaviour', fontsize=11)
ax.set_ylabel('Utilisation Level', fontsize=11)
ax.tick_params(axis='x', labelsize=10, rotation=0)
ax.tick_params(axis='y', labelsize=10, rotation=90)
plt.tight_layout()
plt.savefig(
    'figures/fig17_utilization_repayment_quadrant.png',
    dpi=150, bbox_inches='tight',
)
plt.show()

# ── Quantify the disproportionate-contribution claim ─────────────
high_risk_quadrant = quadrant_df[
    (quadrant_df['util_band'] == UTIL_ORDER[1])
    & (quadrant_df['ratio_band'] == RATIO_ORDER[1])
]
total_defaulters = df[TARGET].sum()
quadrant_defaulters = high_risk_quadrant[TARGET].sum()

print(
    f'High-Utilisation + Poor-Repayment customers: '
    f'{len(high_risk_quadrant):,} '
    f'({len(high_risk_quadrant) / len(df) * 100:.1f}% '
    f'of portfolio)'
)
print(
    f'They account for {quadrant_defaulters:,} defaults '
    f'({quadrant_defaulters / total_defaulters * 100:.1f}% '
    f'of ALL defaults in the dataset)'
)


**Interpretation — Fig 17**  
The bottom-right quadrant confirms the hypothesis: customers with **both** high utilisation and poor repayment default at **29.1%** — the highest of any combination, and roughly double the safest quadrant (15.0%). More importantly, this group is not a small fringe case: at **10,378 customers (34.6% of the entire portfolio)**, they account for **45.5% of every default in the dataset** — a disproportionate concentration of risk relative to their share of the customer base. Notably, high utilisation *alone* (adequate repayment) is comparatively mild (18.4%, only 681 customers) — it is the **combination**, not either factor in isolation, that identifies the highest-value target for intervention.

---
## Section 4.3 — Financial Exposure and Expected Loss by Segment

Addresses the mandate's third question directly: *"where is the company over-exposed... and what is the financial significance of these accounts?"*

**Expected loss** is estimated here as:

```
expected_loss = total_outstanding_balance x observed_default_rate
```

**Stated assumption:** this simplified estimate assumes 100% loss given default (no recovery through collections). A production credit-risk model would apply a recovery-rate haircut; that refinement is out of scope here, but the relative comparison *between segments* remains valid and directionally sound.

In [ ]:
# ── Segment-level financial exposure ──────────────────────────────
exposure_summary = (
    df.groupby('risk_segment', observed=True)
    .agg(
        n=('customer_id', 'count'),
        total_credit_limit=('credit_limit', 'sum'),
        total_outstanding=('outstanding_exposure', 'sum'),
        default_rate=(TARGET, 'mean'),
    )
    .reindex(SEGMENT_ORDER)
)

exposure_summary['expected_loss'] = (
    exposure_summary['total_outstanding']
    * exposure_summary['default_rate']
)
exposure_summary['pct_of_expected_loss'] = (
    exposure_summary['expected_loss']
    / exposure_summary['expected_loss'].sum() * 100
)

# ── Reconciliation check ─────────────────────────────────────────
assert exposure_summary['n'].sum() == len(df), (
    'Segment counts must reconcile to 30,000.'
)

display_summary = exposure_summary.copy()
for col in ['total_credit_limit', 'total_outstanding',
            'expected_loss']:
    display_summary[f'{col}_millions'] = (
        display_summary[col] / 1_000_000
    ).round(1)

print('Financial exposure by segment (NTD millions):')
display_summary[[
    'n', 'total_credit_limit_millions',
    'total_outstanding_millions', 'expected_loss_millions',
    'pct_of_expected_loss',
]].round(1)


In [ ]:
# ── Fig 18: Total outstanding vs expected loss by segment ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
seg_palette = [SEGMENT_COLOURS[s] for s in SEGMENT_ORDER]

# Panel A — total outstanding balance per segment
bars_a = axes[0].bar(
    SEGMENT_ORDER,
    display_summary['total_outstanding_millions'],
    color=seg_palette, edgecolor='white', linewidth=0.8,
)
for bar, val in zip(
    bars_a, display_summary['total_outstanding_millions']
):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, val + 10,
        f'NTD {val:,.1f}M', ha='center', va='bottom',
        fontsize=9, fontweight='bold',
    )
axes[0].set_title(
    'Total Outstanding Balance by Segment',
    fontsize=12, fontweight='bold', pad=10,
)
axes[0].set_ylabel('NTD (millions)', fontsize=10)

# Panel B — estimated expected loss per segment
bars_b = axes[1].bar(
    SEGMENT_ORDER,
    display_summary['expected_loss_millions'],
    color=seg_palette, edgecolor='white', linewidth=0.8,
)
for bar, val, pct in zip(
    bars_b, display_summary['expected_loss_millions'],
    display_summary['pct_of_expected_loss'],
):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, val + 3,
        f'NTD {val:,.1f}M\n({pct:.0f}%)',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
    )
axes[1].set_title(
    'Estimated Expected Loss by Segment',
    fontsize=12, fontweight='bold', pad=10,
)
axes[1].set_ylabel('NTD (millions)', fontsize=10)

fig.suptitle(
    'Fig 18 — Financial Exposure: Balance Carried vs. '
    'Expected Loss',
    fontsize=14, fontweight='bold', y=1.03,
)
plt.tight_layout()
plt.savefig(
    'figures/fig18_financial_exposure.png',
    dpi=150, bbox_inches='tight',
)
plt.show()


**Interpretation — Fig 18 (the key non-obvious finding of this phase)**  
The **Critical** segment has the highest default *rate* (60%), but it is the **At-Risk** segment that carries the largest **expected loss in absolute NTD** — approximately NTD 198.2M, or **53% of the portfolio's entire expected loss**, compared to just 23% from Critical. This happens because At-Risk is more than four times larger than Critical (8,238 vs 1,867 customers) and carries substantially higher outstanding balances per customer.  

**This is the single most important strategic implication of Phase 4:** if FinTrust has limited collections capacity, prioritising outreach purely by default *rate* (i.e. chasing Critical accounts first) would leave the largest pool of actual money at risk unaddressed. **At-Risk deserves equal or greater resourcing priority**, not because its customers are individually riskiest, but because of the segment's sheer size and financial scale.

---
## Section 4.4 — Strategic Action Mapping

Each segment is mapped to a specific, proportionate action via a dictionary. **Ghost/dormant accounts** (identified in Phase 2/3) receive a distinct override: they are technically labelled At-Risk by score, but a credit review or restructuring offer makes no sense for a dormant, zero-balance account. Their action addresses the actual root cause — disengagement, not financial distress.

In [ ]:
# ── Action dictionary: one clear, proportionate action per segment
ACTION_MAP = {
    'Healthy': (
        'Maintain standard servicing. Offer optional loyalty or '
        'rewards incentives to reinforce good repayment habits. '
        'No active risk intervention required.'
    ),
    'Watchlist': (
        'Send automated payment reminders 3-5 days before the due '
        'date. Share financial-wellness resources proactively. '
        'Increase monitoring cadence to monthly.'
    ),
    'At-Risk': (
        'Conduct a credit-limit review (freeze further increases '
        'or reduce limit). Proactively offer a structured '
        'repayment plan. Assign to enhanced monitoring.'
    ),
    'Critical': (
        'Escalate to dedicated case management. Offer '
        'hardship-based restructuring or a settlement plan. '
        'Suspend further credit extension pending review.'
    ),
}

GHOST_ACTION = (
    'Send a dormant-account re-engagement notice (fee-due alert '
    'or account status reminder) before any credit action. '
    'Standard restructuring or limit-review offers are not '
    'appropriate for a zero-balance, inactive account.'
)


def assign_recommended_action(row: pd.Series) -> str:
    """Return the recommended action for a single customer.

    Ghost/dormant accounts are checked first and receive a
    re-engagement action regardless of their scored segment,
    since standard segment actions assume active financial
    behaviour that dormant accounts do not exhibit.

    Parameters
    ----------
    row : pd.Series
        Must contain 'is_ghost_account' and 'risk_segment'.

    Returns
    -------
    str
        The recommended business action.
    """
    if row['is_ghost_account']:
        return GHOST_ACTION
    return ACTION_MAP[row['risk_segment']]


df['recommended_action'] = df.apply(
    assign_recommended_action, axis=1
)

assert df['recommended_action'].notna().all(), (
    'Every customer must receive a recommended action.'
)

# ── Display segment breakdown, separating ghost accounts out ─────
df['segment_display'] = np.where(
    df['is_ghost_account'],
    'At-Risk — Ghost/Dormant',
    df['risk_segment'],
)
DISPLAY_ORDER = [
    'Healthy', 'Watchlist', 'At-Risk',
    'At-Risk — Ghost/Dormant', 'Critical',
]

action_summary = (
    df.groupby('segment_display', observed=True)
    .agg(
        n=('customer_id', 'count'),
        default_rate=(TARGET, 'mean'),
        total_outstanding=('outstanding_exposure', 'sum'),
    )
    .reindex(DISPLAY_ORDER)
)
action_summary['default_rate_pct'] = (
    action_summary['default_rate'] * 100
).round(1)
action_summary['total_outstanding_millions'] = (
    action_summary['total_outstanding'] / 1_000_000
).round(1)

action_summary[[
    'n', 'default_rate_pct', 'total_outstanding_millions'
]]


**Segment → Action Reference Table**

| Segment | n | Default Rate | Recommended Action |
|:--------|--:|-------------:|:--------------------|
| 🟢 Healthy | 9,027 | 13.9% | Standard servicing; optional loyalty rewards |
| 🟡 Watchlist | 10,868 | 15.0% | Automated reminders; financial-wellness content; monthly monitoring |
| 🟠 At-Risk | 7,743 | 31.5%* | Credit-limit review; structured repayment plan; enhanced monitoring |
| 🟣 At-Risk — Ghost/Dormant | 495 | 39.6% | Dormant-account re-engagement notice, **not** restructuring |
| 🔴 Critical | 1,867 | 59.5% | Case management; hardship restructuring; suspend further credit |

*At-Risk default rate shown excluding the Ghost/Dormant sub-population, which is broken out separately above because it requires a fundamentally different action.

**Fairness and transparency note:** every action is grounded in observable repayment behaviour (delinquency, repayment ratio, utilisation, trajectory) — never demographic attributes such as age, sex, education, or marital status. This satisfies the mandate's requirement that interventions remain "fair and transparent."

**A note on the Invisible Defaulters (Phase 2 finding):** the 1,951 customers who defaulted with zero behavioural warning signs are, by construction, indistinguishable from genuinely healthy customers and will be assigned the Healthy or Watchlist action. No behavioural score can target them individually — the appropriate response is portfolio-level (e.g. a standing reserve for unpredictable losses, or the external-data enrichment explored in Phase 5), not a segment-specific action.

---
## Phase 4 Summary

| # | Question | Key Finding |
|--:|:---------|:------------|
| 1 | Does utilisation alone explain risk? | No — credit limit moves opposite to risk; utilisation *relative to* the limit is what matters (Fig 16) |
| 2 | Do high utilisation + weak repayment combine disproportionately? | **Yes** — 34.6% of customers in this combined group account for 45.5% of all defaults (Fig 17) |
| 3 | Which segment holds the most money at risk? | **At-Risk** (53% of expected loss), not Critical (23%) — a sizing effect, not a rate effect (Fig 18) |
| 4 | What should the bank do for each segment? | Four escalating actions, plus a fifth override for ghost/dormant accounts requiring re-engagement instead of restructuring |

**3 visualisations produced** (Figs 16–18), saved to `figures/`.  
**Proceed to Phase 5 — Data Enrichment via API Integration.**